In [41]:
import torch as torch
import pandas as pd
import glob as glob
import matplotlib as plt
import numpy as np
from torch_geometric.data import Data
from sklearn.preprocessing import StandardScaler

from numpy.ma.extras import average
from six import print_
from torch_geometric.utils import is_undirected

In [42]:
# loading dataframe
path = r'C:\Users\hkute\Documents\LearningPython\tennis_predictor\Tennis_Predictor\datasets'

all_files = glob.glob(path + r"\atp_matches_*.csv")

df = pd.concat((pd.read_csv(f) for f in all_files), ignore_index=True)

pd.set_option('display.max_columns', None)

df.head()

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,winner_name,winner_hand,winner_ht,winner_ioc,winner_age,loser_id,loser_seed,loser_entry,loser_name,loser_hand,loser_ht,loser_ioc,loser_age,score,best_of,round,minutes,w_ace,w_df,w_svpt,w_1stIn,w_1stWon,w_2ndWon,w_SvGms,w_bpSaved,w_bpFaced,l_ace,l_df,l_svpt,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points
0,2003-1536,Madrid Masters,Hard,48,M,20031013,1,101965,NaN,NaN,Wayne Ferreira,R,185.0,RSA,32.0,103344,NaN,NaN,Ivan Ljubicic,R,193.0,CRO,24.5,7-6(7) 7-6(5),3,R64,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28.0,1090.0,42.0,865.0
1,2003-1536,Madrid Masters,Hard,48,M,20031013,2,102358,NaN,Q,Thomas Enqvist,R,190.0,SWE,29.5,102338,NaN,NaN,Yevgeny Kafelnikov,R,190.0,RUS,29.6,6-3 RET,3,R64,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,146.0,258.0,40.0,950.0
2,2003-1536,Madrid Masters,Hard,48,M,20031013,3,102998,NaN,Q,Jan Michael Gambill,R,190.0,USA,26.3,103786,NaN,NaN,Nikolay Davydenko,R,178.0,RUS,22.3,6-3 6-3,3,R64,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,57.0,660.0,43.0,855.0
3,2003-1536,Madrid Masters,Hard,48,M,20031013,4,102610,NaN,NaN,Albert Costa,R,180.0,ESP,28.3,103602,NaN,NaN,Fernando Gonzalez,R,183.0,CHI,23.2,6-3 7-6(3),3,R64,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23.0,1170.0,22.0,1190.0
4,2003-1536,Madrid Masters,Hard,48,M,20031013,5,102374,NaN,WC,Alex Corretja,R,180.0,ESP,29.5,104745,NaN,WC,Rafael Nadal,L,185.0,ESP,17.3,6-2 3-6 6-4,3,R64,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,127.0,290.0,49.0,788.0


We will drop some columns

There are missing values which we will need for later on

In [43]:
df.drop(['winner_seed', 'loser_seed','loser_entry', 'winner_entry'], axis=1, inplace=True)
df.head()

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_name,winner_hand,winner_ht,winner_ioc,winner_age,loser_id,loser_name,loser_hand,loser_ht,loser_ioc,loser_age,score,best_of,round,minutes,w_ace,w_df,w_svpt,w_1stIn,w_1stWon,w_2ndWon,w_SvGms,w_bpSaved,w_bpFaced,l_ace,l_df,l_svpt,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points
0,2003-1536,Madrid Masters,Hard,48,M,20031013,1,101965,Wayne Ferreira,R,185.0,RSA,32.0,103344,Ivan Ljubicic,R,193.0,CRO,24.5,7-6(7) 7-6(5),3,R64,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28.0,1090.0,42.0,865.0
1,2003-1536,Madrid Masters,Hard,48,M,20031013,2,102358,Thomas Enqvist,R,190.0,SWE,29.5,102338,Yevgeny Kafelnikov,R,190.0,RUS,29.6,6-3 RET,3,R64,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,146.0,258.0,40.0,950.0
2,2003-1536,Madrid Masters,Hard,48,M,20031013,3,102998,Jan Michael Gambill,R,190.0,USA,26.3,103786,Nikolay Davydenko,R,178.0,RUS,22.3,6-3 6-3,3,R64,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,57.0,660.0,43.0,855.0
3,2003-1536,Madrid Masters,Hard,48,M,20031013,4,102610,Albert Costa,R,180.0,ESP,28.3,103602,Fernando Gonzalez,R,183.0,CHI,23.2,6-3 7-6(3),3,R64,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23.0,1170.0,22.0,1190.0
4,2003-1536,Madrid Masters,Hard,48,M,20031013,5,102374,Alex Corretja,R,180.0,ESP,29.5,104745,Rafael Nadal,L,185.0,ESP,17.3,6-2 3-6 6-4,3,R64,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,127.0,290.0,49.0,788.0


In [44]:
df = df.fillna(df.mean(numeric_only=True))
df

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_name,winner_hand,winner_ht,winner_ioc,winner_age,loser_id,loser_name,loser_hand,loser_ht,loser_ioc,loser_age,score,best_of,round,minutes,w_ace,w_df,w_svpt,w_1stIn,w_1stWon,w_2ndWon,w_SvGms,w_bpSaved,w_bpFaced,l_ace,l_df,l_svpt,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points
0,2003-1536,Madrid Masters,Hard,48,M,20031013,1,101965,Wayne Ferreira,R,185.000000,RSA,32.0,103344,Ivan Ljubicic,R,193.000000,CRO,24.500000,7-6(7) 7-6(5),3,R64,100.565826,5.597889,2.571659,72.170873,44.686945,33.278129,15.119164,11.466324,3.345778,4.929678,4.130268,3.327813,74.848836,45.13172,29.365498,13.441081,11.285491,4.553419,8.399145,28.0,1090.0,42.000000,865.000000
1,2003-1536,Madrid Masters,Hard,48,M,20031013,2,102358,Thomas Enqvist,R,190.000000,SWE,29.5,102338,Yevgeny Kafelnikov,R,190.000000,RUS,29.600000,6-3 RET,3,R64,100.565826,5.597889,2.571659,72.170873,44.686945,33.278129,15.119164,11.466324,3.345778,4.929678,4.130268,3.327813,74.848836,45.13172,29.365498,13.441081,11.285491,4.553419,8.399145,146.0,258.0,40.000000,950.000000
2,2003-1536,Madrid Masters,Hard,48,M,20031013,3,102998,Jan Michael Gambill,R,190.000000,USA,26.3,103786,Nikolay Davydenko,R,178.000000,RUS,22.300000,6-3 6-3,3,R64,100.565826,5.597889,2.571659,72.170873,44.686945,33.278129,15.119164,11.466324,3.345778,4.929678,4.130268,3.327813,74.848836,45.13172,29.365498,13.441081,11.285491,4.553419,8.399145,57.0,660.0,43.000000,855.000000
3,2003-1536,Madrid Masters,Hard,48,M,20031013,4,102610,Albert Costa,R,180.000000,ESP,28.3,103602,Fernando Gonzalez,R,183.000000,CHI,23.200000,6-3 7-6(3),3,R64,100.565826,5.597889,2.571659,72.170873,44.686945,33.278129,15.119164,11.466324,3.345778,4.929678,4.130268,3.327813,74.848836,45.13172,29.365498,13.441081,11.285491,4.553419,8.399145,23.0,1170.0,22.000000,1190.000000
4,2003-1536,Madrid Masters,Hard,48,M,20031013,5,102374,Alex Corretja,R,180.000000,ESP,29.5,104745,Rafael Nadal,L,185.000000,ESP,17.300000,6-2 3-6 6-4,3,R64,100.565826,5.597889,2.571659,72.170873,44.686945,33.278129,15.119164,11.466324,3.345778,4.929678,4.130268,3.327813,74.848836,45.13172,29.365498,13.441081,11.285491,4.553419,8.399145,127.0,290.0,49.000000,788.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
436092,2024-8268,Yokkaichi CH,Hard,32,C,20241125,342,122352,Christoph Negritu,R,193.000000,GER,30.6,206923,Shinji Hazawa,R,175.000000,JPN,25.600000,7-5 6-3,3,Q1,95.000000,5.000000,1.000000,68.000000,39.000000,25.000000,19.000000,11.000000,3.000000,5.000000,0.000000,0.000000,73.000000,54.00000,29.000000,9.000000,10.000000,5.000000,9.000000,409.0,113.0,1073.000000,11.000000
436093,2024-8268,Yokkaichi CH,Hard,32,C,20241125,341,207987,Ryuki Matsuda,R,157.000000,JPN,24.9,200665,Yuta Kawahashi,R,175.000000,JPN,26.800000,6-4 6-0,3,Q1,69.000000,1.000000,0.000000,51.000000,28.000000,20.000000,16.000000,8.000000,5.000000,5.000000,0.000000,1.000000,54.000000,31.00000,16.000000,10.000000,8.000000,4.000000,8.000000,507.0,76.0,860.000000,22.000000
436094,2024-8268,Yokkaichi CH,Hard,32,C,20241125,340,209951,Petr Bar Biryukov,L,196.000000,RUS,22.7,202356,Tsung Hao Huang,R,173.000000,TPE,25.000000,6-4 6-3,3,Q1,67.000000,4.000000,2.000000,62.000000,35.000000,28.000000,7.000000,9.000000,5.000000,8.000000,0.000000,5.000000,66.000000,41.00000,22.000000,8.000000,10.000000,4.000000,10.000000,429.0,100.0,708.000000,40.000000
436095,2024-8268,Yokkaichi CH,Hard,32,C,20241125,338,202113,Hikaru Shiraishi,R,168.000000,JPN,24.5,211627,Hayato Matsuoka,R,180.000000,JPN,19.800000,7-6(6) 3-6 6-4,3,Q1,180.000000,1.000000,2.000000,93.000000,53.000000,29.000000,22.000000,15.000000,2.000000,7.000000,2.000000,1.000000,117.000000,68.00000,41.000000,23.000000,16.000000,7.000000,12.000000,528.0,71.0,894.000000,20.000000


In [45]:
# one hot encoding surface
surface = pd.get_dummies(df.surface, dtype=int)

df = pd.concat([df, surface], axis='columns')
df.drop(["surface"], axis='columns', inplace=True)

df

,tourney_id,tourney_name,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_name,winner_hand,winner_ht,winner_ioc,winner_age,loser_id,loser_name,loser_hand,loser_ht,loser_ioc,loser_age,score,best_of,round,minutes,w_ace,w_df,w_svpt,w_1stIn,w_1stWon,w_2ndWon,w_SvGms,w_bpSaved,w_bpFaced,l_ace,l_df,l_svpt,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points,Carpet,Clay,Grass,Hard
0,2003-1536,Madrid Masters,48,M,20031013,1,101965,Wayne Ferreira,R,185.000000,RSA,32.0,103344,Ivan Ljubicic,R,193.000000,CRO,24.500000,7-6(7) 7-6(5),3,R64,100.565826,5.597889,2.571659,72.170873,44.686945,33.278129,15.119164,11.466324,3.345778,4.929678,4.130268,3.327813,74.848836,45.13172,29.365498,13.441081,11.285491,4.553419,8.399145,28.0,1090.0,42.000000,865.000000,0,0,0,1
1,2003-1536,Madrid Masters,48,M,20031013,2,102358,Thomas Enqvist,R,190.000000,SWE,29.5,102338,Yevgeny Kafelnikov,R,190.000000,RUS,29.600000,6-3 RET,3,R64,100.565826,5.597889,2.571659,72.170873,44.686945,33.278129,15.119164,11.466324,3.345778,4.929678,4.130268,3.327813,74.848836,45.13172,29.365498,13.441081,11.285491,4.553419,8.399145,146.0,258.0,40.000000,950.000000,0,0,0,1
2,2003-1536,Madrid Masters,48,M,20031013,3,102998,Jan Michael Gambill,R,190.000000,USA,26.3,103786,Nikolay Davydenko,R,178.000000,RUS,22.300000,6-3 6-3,3,R64,100.565826,5.597889,2.571659,72.170873,44.686945,33.278129,15.119164,11.466324,3.345778,4.929678,4.130268,3.327813,74.848836,45.13172,29.365498,13.441081,11.285491,4.553419,8.399145,57.0,660.0,43.000000,855.000000,0,0,0,1
3,2003-1536,Madrid Masters,48,M,20031013,4,102610,Albert Costa,R,180.000000,ESP,28.3,103602,Fernando Gonzalez,R,183.000000,CHI,23.200000,6-3 7-6(3),3,R64,100.565826,5.597889,2.571659,72.170873,44.686945,33.278129,15.119164,11.466324,3.345778,4.929678,4.130268,3.327813,74.848836,45.13172,29.365498,13.441081,11.285491,4.553419,8.399145,23.0,1170.0,22.000000,1190.000000,0,0,0,1
4,2003-1536,Madrid Masters,48,M,20031013,5,102374,Alex Corretja,R,180.000000,ESP,29.5,104745,Rafael Nadal,L,185.000000,ESP,17.300000,6-2 3-6 6-4,3,R64,100.565826,5.597889,2.571659,72.170873,44.686945,33.278129,15.119164,11.466324,3.345778,4.929678,4.130268,3.327813,74.848836,45.13172,29.365498,13.441081,11.285491,4.553419,8.399145,127.0,290.0,49.000000,788.000000,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
436092,2024-8268,Yokkaichi CH,32,C,20241125,342,122352,Christoph Negritu,R,193.000000,GER,30.6,206923,Shinji Hazawa,R,175.000000,JPN,25.600000,7-5 6-3,3,Q1,95.000000,5.000000,1.000000,68.000000,39.000000,25.000000,19.000000,11.000000,3.000000,5.000000,0.000000,0.000000,73.000000,54.00000,29.000000,9.000000,10.000000,5.000000,9.000000,409.0,113.0,1073.000000,11.000000,0,0,0,1
436093,2024-8268,Yokkaichi CH,32,C,20241125,341,207987,Ryuki Matsuda,R,157.000000,JPN,24.9,200665,Yuta Kawahashi,R,175.000000,JPN,26.800000,6-4 6-0,3,Q1,69.000000,1.000000,0.000000,51.000000,28.000000,20.000000,16.000000,8.000000,5.000000,5.000000,0.000000,1.000000,54.000000,31.00000,16.000000,10.000000,8.000000,4.000000,8.000000,507.0,76.0,860.000000,22.000000,0,0,0,1
436094,2024-8268,Yokkaichi CH,32,C,20241125,340,209951,Petr Bar Biryukov,L,196.000000,RUS,22.7,202356,Tsung Hao Huang,R,173.000000,TPE,25.000000,6-4 6-3,3,Q1,67.000000,4.000000,2.000000,62.000000,35.000000,28.000000,7.000000,9.000000,5.000000,8.000000,0.000000,5.000000,66.000000,41.00000,22.000000,8.000000,10.000000,4.000000,10.000000,429.0,100.0,708.000000,40.000000,0,0,0,1
436095,2024-8268,Yokkaichi CH,32,C,20241125,338,202113,Hikaru Shiraishi,R,168.000000,JPN,24.5,211627,Hayato Matsuoka,R,180.000000,JPN,19.800000,7-6(6) 3-6 6-4,3,Q1,180.000000,1.000000,2.000000,93.000000,53.000000,29.000000,22.000000,15.000000,2.000000,7.000000,2.000000,1.000000,117.000000,68.00000,41.000000,23.000000,16.000000,7.00000

In [46]:
# separating data to historical and future data
df['tourney_date'] = pd.to_datetime(df['tourney_date'], format='%Y%m%d')

historical_df = df[df['tourney_date'] < "2023-01-01"]
future_df = df[df['tourney_date'] >= "2023-01-01"]

historical_df.head()

,tourney_id,tourney_name,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_name,winner_hand,winner_ht,winner_ioc,winner_age,loser_id,loser_name,loser_hand,loser_ht,loser_ioc,loser_age,score,best_of,round,minutes,w_ace,w_df,w_svpt,w_1stIn,w_1stWon,w_2ndWon,w_SvGms,w_bpSaved,w_bpFaced,l_ace,l_df,l_svpt,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points,Carpet,Clay,Grass,Hard
0,2003-1536,Madrid Masters,48,M,2003-10-13,1,101965,Wayne Ferreira,R,185.0,RSA,32.0,103344,Ivan Ljubicic,R,193.0,CRO,24.5,7-6(7) 7-6(5),3,R64,100.565826,5.597889,2.571659,72.170873,44.686945,33.278129,15.119164,11.466324,3.345778,4.929678,4.130268,3.327813,74.848836,45.13172,29.365498,13.441081,11.285491,4.553419,8.399145,28.0,1090.0,42.0,865.0,0,0,0,1
1,2003-1536,Madrid Masters,48,M,2003-10-13,2,102358,Thomas Enqvist,R,190.0,SWE,29.5,102338,Yevgeny Kafelnikov,R,190.0,RUS,29.6,6-3 RET,3,R64,100.565826,5.597889,2.571659,72.170873,44.686945,33.278129,15.119164,11.466324,3.345778,4.929678,4.130268,3.327813,74.848836,45.13172,29.365498,13.441081,11.285491,4.553419,8.399145,146.0,258.0,40.0,950.0,0,0,0,1
2,2003-1536,Madrid Masters,48,M,2003-10-13,3,102998,Jan Michael Gambill,R,190.0,USA,26.3,103786,Nikolay Davydenko,R,178.0,RUS,22.3,6-3 6-3,3,R64,100.565826,5.597889,2.571659,72.170873,44.686945,33.278129,15.119164,11.466324,3.345778,4.929678,4.130268,3.327813,74.848836,45.13172,29.365498,13.441081,11.285491,4.553419,8.399145,57.0,660.0,43.0,855.0,0,0,0,1
3,2003-1536,Madrid Masters,48,M,2003-10-13,4,102610,Albert Costa,R,180.0,ESP,28.3,103602,Fernando Gonzalez,R,183.0,CHI,23.2,6-3 7-6(3),3,R64,100.565826,5.597889,2.571659,72.170873,44.686945,33.278129,15.119164,11.466324,3.345778,4.929678,4.130268,3.327813,74.848836,45.13172,29.365498,13.441081,11.285491,4.553419,8.399145,23.0,1170.0,22.0,1190.0,0,0,0,1
4,2003-1536,Madrid Masters,48,M,2003-10-13,5,102374,Alex Corretja,R,180.0,ESP,29.5,104745,Rafael Nadal,L,185.0,ESP,17.3,6-2 3-6 6-4,3,R64,100.565826,5.597889,2.571659,72.170873,44.686945,33.278129,15.119164,11.466324,3.345778,4.929678,4.130268,3.327813,74.848836,45.13172,29.365498,13.441081,11.285491,4.553419,8.399145,127.0,290.0,49.0,788.0,0,0,0,1


for Node Features we will combine play results and get an average for each player

In [47]:
# separate winner stats from losers stats
winner_stats = [col for col in historical_df.columns if col.startswith('w_')]
loser_stats = [col for col in historical_df.columns if col.startswith('l_')]

# create winner and loser df
winners = historical_df[['winner_id'] + winner_stats].copy()
winners.columns = ['player_id'] + [col[2:] for col in winner_stats]

losers = historical_df[['loser_id'] + loser_stats].copy()
losers.columns = ['player_id'] + [col[2:] for col in loser_stats]

# combine results
players_df = pd.concat([winners, losers], ignore_index=True)

# get average
player_averages = players_df.groupby('player_id').mean()
player_averages = player_averages.reset_index()

atp_players = pd.read_csv(r'datasets\atp_players.csv', dtype={'wikidata_id':'string'})
#print(atp_players)

atp_players['player_id'] = atp_players['player_id'].astype('Int64')
atp_players['player'] = atp_players['name_first'] + ' ' + atp_players['name_last']


official_df = player_averages.merge(atp_players[['player_id','player','height','dob','hand','ioc']], on='player_id', how='left')
official_df['dob'] = official_df['dob'].astype('Int64')

print(official_df)

       player_id       ace        df       svpt      1stIn     1stWon  \
0         100644  7.575835  3.880909  77.882098  49.684914  36.507129   
1         100653  4.130268  3.327813  74.848836  45.131720  29.365498   
2         101266  4.130268  3.327813  74.848836  45.131720  29.365498   
3         101305  0.000000  5.000000  90.000000  51.000000  31.000000   
4         101316  4.130268  3.327813  74.848836  45.131720  29.365498   
...          ...       ...       ...        ...        ...        ...   
12712     211773  4.130268  3.327813  74.848836  45.131720  29.365498   
12713     211774  5.108682  2.823710  73.063527  44.835203  31.973919   
12714     211776  2.500000  3.000000  56.500000  35.000000  20.500000   
12715     212813  4.130268  3.327813  74.848836  45.131720  29.365498   
12716     212814  4.130268  3.327813  74.848836  45.131720  29.365498   

          2ndWon      SvGms    bpSaved    bpFaced                  player  \
0      14.191667  12.486935   3.564218   5.884

In [48]:
# make sure there is only one player per row
print(max(official_df['player_id'].value_counts()))

1


In [49]:
sorted_df = official_df.sort_values(by='player_id')

sorted_df

,player_id,ace,df,svpt,1stIn,1stWon,2ndWon,SvGms,bpSaved,bpFaced,player,height,dob,hand,ioc
0,100644,7.575835,3.880909,77.882098,49.684914,36.507129,14.191667,12.486935,3.564218,5.884390,Alexander Zverev,198.0,19970420,R,GER
1,100653,4.130268,3.327813,74.848836,45.131720,29.365498,13.441081,11.285491,4.553419,8.399145,Andres Gomez,193.0,19600227,L,ECU
2,101266,4.130268,3.327813,74.848836,45.131720,29.365498,13.441081,11.285491,4.553419,8.399145,Trey Carter,188.0,19660702,R,USA
3,101305,0.000000,5.000000,90.000000,51.000000,31.000000,18.000000,10.000000,10.000000,14.000000,Jeff Greenwald,NaN,19661110,R,USA
4,101316,4.130268,3.327813,74.848836,45.131720,29.365498,13.441081,11.285491,4.553419,8.399145,Juan Rios,185.0,19661215,R,PUR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12712,211773,4.130268,3.327813,74.848836,45.131720,29.365498,13.441081,11.285491,4.553419,8.399145,Mohamed El Hafedh Said,NaN,<NA>,U,MRT
12713,211774,5.108682,2.823710,73.063527,44.835203,31.973919,14.559803,11.406046,3.748325,6.086167,Ryotaro Taguchi,NaN,20000630,U,JPN
12714,211776,2.500000,3.000000,56.500000,35.000000,20.500000,7.000000,8.000000,4.500000,9.000000,Martin Landaluce,191.0,20060108,R,ESP
12715,212813,4.130268,3.327813,74.848836,45.131720,29.365498,13.441081,11.285491,4.553419,8.399145,Kentaro Otsuka,NaN,<NA>,U,JPN


In [50]:
# load node features
node_features = sorted_df[['ace','df','svpt','1stIn','1stWon','2ndWon', 'SvGms','bpSaved','bpFaced','height','dob','hand']]

node_features

,ace,df,svpt,1stIn,1stWon,2ndWon,SvGms,bpSaved,bpFaced,height,dob,hand
0,7.575835,3.880909,77.882098,49.684914,36.507129,14.191667,12.486935,3.564218,5.884390,198.0,19970420,R
1,4.130268,3.327813,74.848836,45.131720,29.365498,13.441081,11.285491,4.553419,8.399145,193.0,19600227,L
2,4.130268,3.327813,74.848836,45.131720,29.365498,13.441081,11.285491,4.553419,8.399145,188.0,19660702,R
3,0.000000,5.000000,90.000000,51.000000,31.000000,18.000000,10.000000,10.000000,14.000000,NaN,19661110,R
4,4.130268,3.327813,74.848836,45.131720,29.365498,13.441081,11.285491,4.553419,8.399145,185.0,19661215,R
...,...,...,...,...,...,...,...,...,...,...,...,...
12712,4.130268,3.327813,74.848836,45.131720,29.365498,13.441081,11.285491,4.553419,8.399145,NaN,<NA>,U
12713,5.108682,2.823710,73.063527,44.835203,31.973919,14.559803,11.406046,3.748325,6.086167,NaN,20000630,U
12714,2.500000,3.000000,56.500000,35.000000,20.500000,7.000000,8.000000,4.500000,9.000000,191.0,20060108,R
12715,4.130268,3.327813,74.848836,45.131720,29.365498,13.441081,11.285491,4.553419,8.399145,NaN,<NA>,U


In [51]:
#convert non-numeric vals/one hot encoding for dominant hand

pd.set_option('mode.chained_assignment', None)
#handness = node_features["hand"].str.split(",", expand=True)
hands = pd.get_dummies(node_features.hand, dtype=int)

node_features = pd.concat([node_features, hands], axis='columns')
node_features.drop(["hand"], axis='columns', inplace=True)

In [52]:
# transforming dob into an age and filling missing values

node_features['age'] = 2026 - pd.to_datetime(node_features['dob'].astype(str), format='%Y%m%d', errors='coerce').dt.year
node_features['age'] = node_features['age'].fillna(node_features['age'].mean())
node_features['height'] = node_features['height'].fillna(node_features['height'].mean())
node_features = node_features.drop(['dob'], axis=1)
node_features

,ace,df,svpt,1stIn,1stWon,2ndWon,SvGms,bpSaved,bpFaced,height,A,L,R,U,age
0,7.575835,3.880909,77.882098,49.684914,36.507129,14.191667,12.486935,3.564218,5.884390,198.000000,0,0,1,0,29.000000
1,4.130268,3.327813,74.848836,45.131720,29.365498,13.441081,11.285491,4.553419,8.399145,193.000000,0,1,0,0,66.000000
2,4.130268,3.327813,74.848836,45.131720,29.365498,13.441081,11.285491,4.553419,8.399145,188.000000,0,0,1,0,60.000000
3,0.000000,5.000000,90.000000,51.000000,31.000000,18.000000,10.000000,10.000000,14.000000,183.874139,0,0,1,0,60.000000
4,4.130268,3.327813,74.848836,45.131720,29.365498,13.441081,11.285491,4.553419,8.399145,185.000000,0,0,1,0,60.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12712,4.130268,3.327813,74.848836,45.131720,29.365498,13.441081,11.285491,4.553419,8.399145,183.874139,0,0,0,1,32.815917
12713,5.108682,2.823710,73.063527,44.835203,31.973919,14.559803,11.406046,3.748325,6.086167,183.874139,0,0,0,1,26.000000
12714,2.500000,3.000000,56.500000,35.000000,20.500000,7.000000,8.000000,4.500000,9.000000,191.000000,0,0,1,0,20.000000
12715,4.130268,3.327813,74.848836,45.131720,29.365498,13.441081,11.285491,4.553419,8.399145,183.874139,0,0,0,1,32.815917


In [53]:
sc = StandardScaler()
cols_to_scale = ['ace','df','svpt','1stIn','1stWon','2ndWon','SvGms','bpSaved','bpFaced','height','age']
node_features[cols_to_scale] = sc.fit_transform(node_features[cols_to_scale])

node_features

,ace,df,svpt,1stIn,1stWon,2ndWon,SvGms,bpSaved,bpFaced,height,A,L,R,U,age
0,3.381688,1.011747,0.863078,1.417194,2.089253,0.447848,1.703745,-1.194877,-1.858583,3.076321e+00,0,0,1,0,-5.583396e-01
1,-0.148743,0.140564,0.311357,0.239750,-0.017352,-0.044809,0.175442,0.458939,0.681829,1.987424e+00,0,1,0,0,4.855448e+00
2,-0.148743,0.140564,0.311357,0.239750,-0.017352,-0.044809,0.175442,0.458939,0.681829,8.985273e-01,0,0,1,0,3.977536e+00
3,-4.380742,2.774435,3.067206,1.757272,0.464785,2.947497,-1.459775,9.564915,6.339825,-6.189662e-15,0,0,1,0,3.977536e+00
4,-0.148743,0.140564,0.311357,0.239750,-0.017352,-0.044809,0.175442,0.458939,0.681829,2.451892e-01,0,0,1,0,3.977536e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12712,-0.148743,0.140564,0.311357,0.239750,-0.017352,-0.044809,0.175442,0.458939,0.681829,-6.189662e-15,0,0,0,1,1.039656e-15
12713,0.853769,-0.653452,-0.013373,0.163071,0.752067,0.689479,0.328794,-0.887073,-1.654747,-6.189662e-15,0,0,0,1,-9.972953e-01
12714,-1.819166,-0.375777,-3.026117,-2.380287,-2.632457,-4.272496,-4.003885,0.369630,1.288814,1.551865e+00,0,0,1,0,-1.875207e+00
12715,-0.148743,0.140564,0.311357,0.239750,-0.017352,-0.044809,0.175442,0.458939,0.681829,-6.189662e-15,0,0,0,1,1.039656e-15


In [54]:
# Convert to numpy
x = node_features.to_numpy()
print(x.shape)

(12717, 15)


In [55]:
# node index for players to be recognised by row
sorted_df = sorted_df.reset_index(drop=True)

player_id_to_idx = {
    player_id: idx
    for idx, player_id in enumerate(sorted_df['player_id'])
}

print(player_id_to_idx)
print(len(player_id_to_idx))

{100644: 0, 100653: 1, 101266: 2, 101305: 3, 101316: 4, 101339: 5, 101389: 6, 101404: 7, 101481: 8, 101492: 9, 101495: 10, 101509: 11, 101532: 12, 101535: 13, 101543: 14, 101549: 15, 101662: 16, 101723: 17, 101736: 18, 101746: 19, 101750: 20, 101772: 21, 101774: 22, 101775: 23, 101779: 24, 101790: 25, 101806: 26, 101810: 27, 101820: 28, 101855: 29, 101868: 30, 101869: 31, 101871: 32, 101885: 33, 101889: 34, 101895: 35, 101897: 36, 101924: 37, 101938: 38, 101947: 39, 101956: 40, 101960: 41, 101962: 42, 101964: 43, 101965: 44, 101976: 45, 101979: 46, 101990: 47, 101996: 48, 102021: 49, 102023: 50, 102031: 51, 102033: 52, 102035: 53, 102055: 54, 102060: 55, 102061: 56, 102076: 57, 102077: 58, 102087: 59, 102093: 60, 102100: 61, 102106: 62, 102107: 63, 102110: 64, 102123: 65, 102131: 66, 102143: 67, 102148: 68, 102167: 69, 102169: 70, 102179: 71, 102184: 72, 102202: 73, 102207: 74, 102223: 75, 102227: 76, 102231: 77, 102233: 78, 102234: 79, 102240: 80, 102243: 81, 102247: 82, 102250: 83, 1

Edges For Graphs

In [56]:
#
edge_source = historical_df['winner_id'].map(player_id_to_idx).values
edge_target = historical_df['loser_id'].map(player_id_to_idx).values

edge_index_np = np.stack([edge_source, edge_target],axis=0)
edge_index = torch.tensor(edge_index_np, dtype=torch.long)
print(edge_index.shape)

torch.Size([2, 372133])


In [57]:
# build data object
data = Data(x=x, edge_index=edge_index)
print(data)

Data(x=[12717, 15], edge_index=[2, 372133])


In [58]:
from torch_geometric.transforms import ToUndirected
data = ToUndirected()(data)

data

Data(x=[12717, 15], edge_index=[2, 555189])

In [59]:
import torch_geometric.transforms as T

transform = T.RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    is_undirected=True,
    add_negative_train_samples=True,
    neg_sampling_ratio=1.0,
)
train_data, val_data, test_data = transform(data)

print(train_data)
print(val_data)
print(test_data)




Data(x=[12717, 15], edge_index=[2, 444156], edge_label=[444156], edge_label_index=[2, 444156])
Data(x=[12717, 15], edge_index=[2, 444156], edge_label=[55518], edge_label_index=[2, 55518])
Data(x=[12717, 15], edge_index=[2, 499674], edge_label=[55518], edge_label_index=[2, 55518])


In [60]:
# saving data to be used on gnn
torch.save(data, 'atp_graph.pt')
torch.save(train_data, 'train_data.pt')
torch.save(val_data, 'val_data.pt')
torch.save(test_data, 'test_data.pt')